# TB Portals - **Agentic Rung 1**: RAD-DINO features + Balanced MSE

Phase 0 (cache frozen RAD-DINO embeddings once) + Rung 1 (light heads on cached features). Isolates the two highest-evidence floor-raisers **before** any agentic machinery: a CXR-foundation backbone + a balanced-regression loss + honest (val-Pearson, all-epoch) selection.

Mode **a2**: ALP head + cavity head -> Timika. Compared to BOTH our locked baseline (honest target) and Kantipudi (aspirational).

| | Romania | **Moldova** | Kazakhstan |
|---|---|---|---|
| locked A2 (beat this) | 20.11 | **30.68** | 21.35 |
| Kantipudi A2 (aspirational) | 18.70 | 18.85 | 19.62 |

**Attach dataset:** `tb-portals-cxr-pngs` only. **Internet: ON** (RAD-DINO downloads from Hugging Face; it is NOT gated). No MedSAM / crops / TBX needed.

## 0 - Clone  *(restart kernel after any pull that changed .py)*

In [1]:
import os, sys, subprocess
REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"
BRANCH   = "cleaned-repo"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
for _p in (REPO_DIR, REPO_DIR + "/scripts"):
    if _p not in sys.path: sys.path.insert(0, _p)
print("repo ready at", REPO_DIR)
# After a git pull that changed .py modules, RESTART the kernel so Python reloads them.

Cloning into '/kaggle/working/dl-project-codebase'...


repo ready at /kaggle/working/dl-project-codebase


Updating files: 100% (453/453), done.


## Install deps (transformers for RAD-DINO; torchxrayvision = fallback backbone)

In [2]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "torchxrayvision", "pydicom", "pylibjpeg", "pylibjpeg-libjpeg"], check=False)
print("deps installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.0 MB/s eta 0:00:00
deps installed


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

## Paths

In [3]:
import os
WORK          = "/kaggle/working"
REPO_DIR      = "/kaggle/working/dl-project-codebase"
DATASET       = "/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs"
KAGGLE_EXPORT = f"{DATASET}/kaggle_export"
PAPER_MANIFEST = f"{WORK}/tbportals_manifest_paper.csv"
BACKBONE      = "rad-dino"                       # rad-dino (primary) | txrv (fallback) | densenet (control)
FEATURES      = f"{WORK}/features_{BACKBONE}.npz"
print("KAGGLE_EXPORT:", KAGGLE_EXPORT, "->", os.path.isdir(KAGGLE_EXPORT))
print("backbone:", BACKBONE, "-> features cache:", FEATURES)

KAGGLE_EXPORT: /kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs/kaggle_export -> True
backbone: rad-dino -> features cache: /kaggle/working/features_rad-dino.npz


## 1 - Build the 5,010-image manifest (Kantipudi Table 1)

In [4]:
import sys, pandas as pd
from pathlib import Path
if REPO_DIR + "/scripts" not in sys.path: sys.path.insert(0, REPO_DIR + "/scripts")
from build_paper_manifest import subsample, PAPER_TOTAL
raw = pd.read_csv(f"{KAGGLE_EXPORT}/manifest.csv",
                  dtype={"image_id": str, "patient_id": str, "country": str})
raw["image_path"] = raw["image_path"].apply(
    lambda p: p if str(p).startswith("/") else f"{KAGGLE_EXPORT}/{p}")
paper_df = subsample(raw, seed=42)
paper_df["image_id"] = paper_df["image_path"].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print(f"Paper manifest: {len(paper_df)} images (target {PAPER_TOTAL}) -> {PAPER_MANIFEST}")

country       need_c  have_c  need_n  have_n
Georgia          713    1298     701    1108
Belarus          254     285     798     894
Ukraine          500    1206     816    1775
Kazakhstan       159     304     240     384
Romania          143     233      77     171
Moldova          193     278     396     527
Azerbaijan         5       7      12      18
India              0       6       3      12
Paper manifest: 5010 images (target 5010) -> /kaggle/working/tbportals_manifest_paper.csv


## 2 - Phase 0: cache frozen RAD-DINO features (~10-20 min, one-time)

In [5]:
# Phase 0: cache FROZEN RAD-DINO features once (~10-20 min, downloads model from HF).
# Idempotent: skips if the cache already exists.
import os, sys
if REPO_DIR + "/scripts" not in sys.path: sys.path.insert(0, REPO_DIR + "/scripts")
if os.path.isfile(FEATURES):
    print("features already cached ->", FEATURES)
else:
    from cache_features import main as cache_main
    cache_main(["--manifest", PAPER_MANIFEST, "--out", FEATURES,
                "--backbone", BACKBONE, "--batch-size", "32"])

[cache] backbone=rad-dino device=cuda


preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[cache] feature dim = 768
[cache] 320/5010
[cache] 640/5010
[cache] 960/5010
[cache] 1280/5010
[cache] 1600/5010
[cache] 1920/5010
[cache] 2240/5010
[cache] 2560/5010
[cache] 2880/5010
[cache] 3200/5010
[cache] 3520/5010
[cache] 3840/5010
[cache] 4160/5010
[cache] 4480/5010
[cache] 4800/5010
[cache] wrote 5010 x 768 features -> /kaggle/working/features_rad-dino.npz


## 3 - Rung 1a: backbone + plain MSE

In [6]:
# Rung 1a: RAD-DINO features + plain-MSE head (isolates the BACKBONE lift).
from src.training.train_agentic import main as agentic_main
agentic_main(["--features", FEATURES, "--manifest", PAPER_MANIFEST,
              "--out-dir", f"{WORK}/checkpoints/agentic_a2_mse", "--mode", "a2",
              "--loss", "mse", "--select-metric", "pearson",
              "--held-outs", "Romania", "Moldova", "Kazakhstan", "--seeds", "0", "1", "2"])

[agentic] device=cuda backbone=rad-dino dim=768 loss=mse select=pearson tag=rad-dino_mse

===== agentic[rad-dino_mse]  Romania  seed=0 =====
[tbportals] split held_out=Romania: train=3832 val=958 test=220 (train/val patients 3618/904).
[RESULT] Romania s0  Timika_MAE=19.63 CI95=[17.40,21.90]  Pearson=0.662 | base 20.11/0.68 | paper 18.70  -> within noise of baseline

===== agentic[rad-dino_mse]  Romania  seed=1 =====
[tbportals] split held_out=Romania: train=3836 val=954 test=220 (train/val patients 3618/904).
[RESULT] Romania s1  Timika_MAE=21.32 CI95=[19.09,23.72]  Pearson=0.630 | base 20.11/0.68 | paper 18.70  -> within noise of baseline

===== agentic[rad-dino_mse]  Romania  seed=2 =====
[tbportals] split held_out=Romania: train=3843 val=947 test=220 (train/val patients 3618/904).
[RESULT] Romania s2  Timika_MAE=20.23 CI95=[18.11,22.47]  Pearson=0.650 | base 20.11/0.68 | paper 18.70  -> within noise of baseline

===== agentic[rad-dino_mse]  Moldova  seed=0 =====
[tbportals] split h

## 4 - Rung 1b: + Balanced MSE (the Moldova-targeted loss)

In [7]:
# Rung 1b: + Balanced MSE (isolates the LOSS fix for the sicker held-out country).
from src.training.train_agentic import main as agentic_main
agentic_main(["--features", FEATURES, "--manifest", PAPER_MANIFEST,
              "--out-dir", f"{WORK}/checkpoints/agentic_a2_bmc", "--mode", "a2",
              "--loss", "bmc", "--select-metric", "pearson",
              "--held-outs", "Romania", "Moldova", "Kazakhstan", "--seeds", "0", "1", "2"])

[agentic] device=cuda backbone=rad-dino dim=768 loss=bmc select=pearson tag=rad-dino_bmc

===== agentic[rad-dino_bmc]  Romania  seed=0 =====
[tbportals] split held_out=Romania: train=3832 val=958 test=220 (train/val patients 3618/904).
[RESULT] Romania s0  Timika_MAE=20.61 CI95=[18.37,22.86]  Pearson=0.653 | base 20.11/0.68 | paper 18.70  -> within noise of baseline

===== agentic[rad-dino_bmc]  Romania  seed=1 =====
[tbportals] split held_out=Romania: train=3836 val=954 test=220 (train/val patients 3618/904).
[RESULT] Romania s1  Timika_MAE=20.68 CI95=[18.45,23.12]  Pearson=0.624 | base 20.11/0.68 | paper 18.70  -> within noise of baseline

===== agentic[rad-dino_bmc]  Romania  seed=2 =====
[tbportals] split held_out=Romania: train=3843 val=947 test=220 (train/val patients 3618/904).
[RESULT] Romania s2  Timika_MAE=20.31 CI95=[18.15,22.49]  Pearson=0.651 | base 20.11/0.68 | paper 18.70  -> within noise of baseline

===== agentic[rad-dino_bmc]  Moldova  seed=0 =====
[tbportals] split h

## 5 - Save (download -> baseline_runs/agentic/)

In [8]:
import os, shutil
dst = f"{WORK}/agentic_rung1"
os.makedirs(dst, exist_ok=True)
for tag in ("mse", "bmc"):
    d = f"{WORK}/checkpoints/agentic_a2_{tag}"
    if os.path.isdir(d):
        shutil.copy(f"{d}/results_agentic.csv", f"{dst}/results_agentic_{tag}.csv")
shutil.copy(FEATURES, f"{dst}/{os.path.basename(FEATURES)}")   # reuse the cache next time
zip_path = shutil.make_archive(f"{WORK}/agentic_rung1", "zip", dst)
print("Saved ->", zip_path)
print("Download it; drop the two results CSVs into baseline_runs/agentic/ and keep the .npz to skip re-caching.")

Saved -> /kaggle/working/agentic_rung1.zip
Download it; drop the two results CSVs into baseline_runs/agentic/ and keep the .npz to skip re-caching.
